In [2]:
import sqlite3
import numpy as np
import pandas as pd
import os

In [86]:
SQL_FOOD_NUTRIENT_ALL = """
    SELECT 
        fn.fdc_id,
        fn.nutrient_id,
        n.nutrient_name,
        fn.amount,
        n.nutrient_unit,
        f.description,
        f.category_id,
        f.category,
        f.data_source,
        f.publication_date
    FROM food_nutrient fn
    JOIN food f ON fn.fdc_id = f.fdc_id
    JOIN nutrient n ON fn.nutrient_id = n.nutrient_id
"""

SQL_FOOD_NUTRIENT_ARG = """
    SELECT 
        fn.fdc_id,
        fn.nutrient_id,
        n.nutrient_name,
        fn.amount,
        n.nutrient_unit,
        f.description,
        f.category_id,
        f.category,
        f.data_source,
        f.publication_date
    FROM food_nutrient fn
    JOIN food f ON fn.fdc_id = f.fdc_id
    JOIN nutrient n ON fn.nutrient_id = n.nutrient_id
    WHERE fn.fdc_id IN (%REPLACE%);
"""


def select_food_nutrient(con:sqlite3.Connection, fdc_ids:list[int]) -> list:
    match fdc_ids:
        case None:
            query = SQL_FOOD_NUTRIENT_ALL
            return pd.read_sql(con=con, sql=query)
        case [*items]:
            placeholders = ",".join(["?"] * len(items))
            query = SQL_FOOD_NUTRIENT_ARG.replace("%REPLACE%", placeholders)
            return pd.read_sql(con=con, sql=query, params=items)
        case _: raise ValueError

SQL_FOOD_ALL = "SELECT * FROM food"
SQL_FOOD_ARG = "SELECT * FROM food WHERE fdc_id IN (%REPLACE%)"

def select_food(con: sqlite3.Connection, fdc_ids: list[int]) -> list:
    match fdc_ids:
        case None:
            query = SQL_FOOD_ALL
            return pd.read_sql(con=con, sql=query)
        case [*items]:
            placeholders = ",".join(["?"] * len(items))
            query = SQL_FOOD_ARG.replace("%REPLACE%", placeholders)
            return pd.read_sql(con=con, sql=query, params=items)
        case _: raise ValueError

def select_nutrient(con: sqlite3.Connection, fdc_ids: list[int]) -> list:
    if not fdc_ids: return []
    
    placeholders = ",".join(["?"] * len(fdc_ids))

    query = f"""
        SELECT * 
        FROM nutrient
        WHERE nutrient_id IN ({placeholders});
    """

    return pd.read_sql(con=con, sql=query, params=fdc_ids)


In [88]:
con = sqlite3.connect("../../data/db/food.db")
con.execute("PRAGMA foreign_key = ON;")
res = select_food(con, None)
con.close()
res

,fdc_id,data_source,publication_date,description,category_id,category
0,167512,sr_legacy_food,2019-04-01,"Pillsbury Golden Layer Buttermilk Biscuits, Ar...",18,Baked Products
1,167513,sr_legacy_food,2019-04-01,"Pillsbury, Cinnamon Rolls with Icing, refriger...",18,Baked Products
2,167514,sr_legacy_food,2019-04-01,"Kraft Foods, Shake N Bake Original Recipe, Coa...",18,Baked Products
3,167515,sr_legacy_food,2019-04-01,"George Weston Bakeries, Thomas English Muffins",18,Baked Products
4,167516,sr_legacy_food,2019-04-01,"Waffles, buttermilk, frozen, ready-to-heat",18,Baked Products
...,...,...,...,...,...,...
8065,2747672,foundation_food,2025-12-18,"Swordfish, frozen, wild caught",15,Finfish and Shellfish Products
8066,2747673,foundation_food,2025-12-18,"Tuna, ahi or yellowfin, frozen, wild caught",15,Finfish and Shellfish Products
8067,2747674,foundation_food,2025-12-18,"Turnips, raw",11,Vegetables and Vegetable Products
8068,2747675,foundation_food,2025-12-18,"Watermelon, seedless, flesh only, raw",9,Fruits and Fruit Juices


In [ ]:
# def insert(table_name:str, col_names:list, rows:list[list], con:sqlite3.Connection) -> None:
#     cur = con.cursor()
#     cols_sql = ', '.join(col_names)
#     params_sql = ','.join(['?' for _ in col_names])
#     sql_insert = f"INSERT INTO {table_name} ({cols_sql}) VALUES ({params_sql})"
#     cur.executemany(sql_insert, rows)
#     cur.close()
#     con.commit()


array([], shape=(0, 0), dtype=float64)